# 📦 Module 01 — Data Formats for Agentic AI
## TOML · YAML · JSON · JSONL

> **Marevlo AI Platform** · All Levels: Beginner → Expert

---
### What you'll learn
- How JSON, YAML, TOML, and JSONL differ and when to use each
- How these formats power real AI agent pipelines
- Hands-on parsing, writing, and validating each format in Python
- Production patterns: streaming, schema validation, CI/CD integration

---
### Quick Install

In [ ]:
# Install required packages
!pip install pyyaml tomli-w pydantic orjson jsonpatch -q

---
## 📋 Part 1 — JSON
### 🟢 Beginner: Reading & Writing JSON

In [ ]:
import json

# --- The 6 JSON data types ---
agent_config = {
    "name": "MIRA",           # string
    "version": 2,              # number (int)
    "temperature": 0.7,        # number (float)
    "active": True,            # boolean
    "context": None,           # null
    "tools": ["search", "memory", "calculator"],  # array
    "model": {                 # object (nested)
        "provider": "anthropic",
        "name": "claude-opus-4-5",
        "max_tokens": 4096
    }
}

# Serialize to JSON string
json_str = json.dumps(agent_config, indent=2)
print("=== JSON String ===")
print(json_str)

In [ ]:
# Deserialize back to Python dict
parsed = json.loads(json_str)

# Accessing nested values
print(f"Agent name: {parsed['name']}")
print(f"Model: {parsed['model']['name']}")
print(f"First tool: {parsed['tools'][0]}")
print(f"Type of 'active': {type(parsed['active'])}")

In [ ]:
import tempfile, os

# Write to file
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump(agent_config, f, indent=2)
    tmppath = f.name

# Read from file
with open(tmppath) as f:
    loaded = json.load(f)

print(f"Loaded from file: {loaded['name']} with {len(loaded['tools'])} tools")
os.unlink(tmppath)

### 🟡 Intermediate: LLM Tool Call Schemas

In [ ]:
# Define an Anthropic-style tool schema
telemetry_tool = {
    "name": "get_device_telemetry",
    "description": "Retrieve real-time telemetry metrics from a network device",
    "input_schema": {
        "type": "object",
        "properties": {
            "device_id": {
                "type": "string",
                "description": "Device identifier (e.g. JNP-001)"
            },
            "metric": {
                "type": "string",
                "enum": ["cpu_util", "mem_util", "pfe_errors", "temp_c"],
                "description": "The metric to retrieve"
            },
            "window_hours": {
                "type": "integer",
                "default": 24,
                "description": "Lookback window in hours"
            }
        },
        "required": ["device_id", "metric"]
    }
}

print(json.dumps(telemetry_tool, indent=2))

In [ ]:
# Simulating parsing an LLM response with tool use
mock_response_content = [
    {"type": "text", "text": "I'll check the telemetry for that device."},
    {
        "type": "tool_use",
        "id": "toolu_01XF",
        "name": "get_device_telemetry",
        "input": {"device_id": "JNP-001", "metric": "cpu_util", "window_hours": 6}
    }
]

for block in mock_response_content:
    if block["type"] == "tool_use":
        tool_name = block["name"]
        tool_input = block["input"]
        print(f"Tool called: {tool_name}")
        print(f"Device: {tool_input['device_id']}")
        print(f"Metric: {tool_input['metric']}")
        print(f"Window: {tool_input['window_hours']}h")

In [ ]:
# PITFALL: LLMs sometimes return JSON wrapped in markdown fences
llm_output_with_fences = '''
Sure! Here's the JSON:
```json
{"device": "JNP-001", "status": "critical", "score": 0.92}
```
'''

def safe_parse_llm_json(text: str) -> dict:
    """Safely extract and parse JSON from LLM output."""
    # Strip markdown fences
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        # Remove first line (```json or ```) and last line (```)
        text = "\n".join(lines[1:-1]).strip()
    return json.loads(text)

result = safe_parse_llm_json(llm_output_with_fences)
print(f"Parsed: {result}")
print(f"Status: {result['status']}")

### 🔴 Advanced: Agent Memory + Streaming JSON

In [ ]:
from dataclasses import dataclass, asdict
from typing import Literal

@dataclass
class Message:
    role: Literal["user", "assistant", "tool"]
    content: str | list
    tool_call_id: str | None = None

class AgentMemory:
    """Manages conversation history with token budget."""
    
    def __init__(self, max_chars: int = 50_000):
        self.messages: list[Message] = []
        self.max_chars = max_chars

    def add(self, msg: Message):
        self.messages.append(msg)
        self._trim_to_budget()

    def _size(self, msg: Message) -> int:
        return len(json.dumps(asdict(msg)))

    def _trim_to_budget(self):
        total = sum(self._size(m) for m in self.messages)
        while total > self.max_chars and len(self.messages) > 2:
            removed = self.messages.pop(1)  # Keep index 0 (system)
            total -= self._size(removed)
            print(f"  [memory] Evicted message to stay within budget")

    def to_api_format(self) -> list[dict]:
        return [asdict(m) for m in self.messages]

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump(self.to_api_format(), f, indent=2)

# Demo
memory = AgentMemory(max_chars=1000)  # Small limit to demo eviction
memory.add(Message(role="user", content="Check JNP-001"))
memory.add(Message(role="assistant", content="Fetching telemetry now..."))
memory.add(Message(role="user", content="What about JNP-002?"))
memory.add(Message(role="assistant", content="Analyzing JNP-002 - CPU is at 78%"))
memory.add(Message(role="user", content="Compare both devices"))

print(f"Messages in memory: {len(memory.messages)}")
print(json.dumps(memory.to_api_format(), indent=2))

### 🟣 Expert: orjson + msgspec for Production Speed

In [ ]:
import orjson
import time
import numpy as np

# Generate large dataset
large_data = [
    {"device_id": f"JNP-{i:03d}", "cpu_util": float(np.random.uniform(20, 99)),
     "mem_util": float(np.random.uniform(30, 95)), "timestamp": 1710480000 + i}
    for i in range(10_000)
]

# Benchmark stdlib vs orjson
t0 = time.perf_counter()
for _ in range(10):
    s = json.dumps(large_data)
    d = json.loads(s)
stdlib_time = (time.perf_counter() - t0) / 10

t0 = time.perf_counter()
for _ in range(10):
    s = orjson.dumps(large_data)
    d = orjson.loads(s)
orjson_time = (time.perf_counter() - t0) / 10

print(f"stdlib json:  {stdlib_time*1000:.2f}ms per 10k records")
print(f"orjson:       {orjson_time*1000:.2f}ms per 10k records")
print(f"Speedup: {stdlib_time/orjson_time:.1f}x")

In [ ]:
# orjson handles numpy arrays natively — stdlib json.dumps() would fail!
telemetry_array = {
    "device": "JNP-001",
    "cpu_readings": np.array([45.2, 67.8, 92.1, 88.4, 91.0]),
    "anomaly_flags": np.array([False, False, True, True, True])
}

try:
    json.dumps(telemetry_array)  # This will fail
except TypeError as e:
    print(f"stdlib fails: {e}")

# orjson handles it natively
serialized = orjson.dumps(telemetry_array, option=orjson.OPT_SERIALIZE_NUMPY)
print(f"orjson succeeds: {serialized[:80]}...")

---
## 📄 Part 2 — YAML
### 🟢 Beginner: YAML Basics

In [ ]:
import yaml

# Writing YAML from Python dict
agent_def = {
    "agent": {
        "name": "MIRA",
        "model": "claude-opus-4-5",
        "temperature": 0.7,
        "tools": ["search", "memory", "calculator"],
        "prompt": "You are a helpful AI tutor on the Marevlo platform."
    }
}

yaml_str = yaml.dump(agent_def, default_flow_style=False, sort_keys=False)
print("=== YAML Output ===")
print(yaml_str)

In [ ]:
# CRITICAL: The Norway Problem
dangerous_yaml = """
country: NO
flag: YES
pi: 3.14
launch_date: 2024-03-15
api_key: "true"
"""

parsed = yaml.safe_load(dangerous_yaml)
print("Parsed values and their types:")
for k, v in parsed.items():
    print(f"  {k}: {repr(v):20s} -> {type(v).__name__}")

### 🟡 Intermediate: Loading Agent Configs from YAML

In [ ]:
import tempfile

# Write a multi-agent YAML config
crewai_config = """
# Multi-agent system config for Marevlo content team
researcher:
  role: Senior ML Researcher
  goal: Find and summarize latest papers on time-series forecasting
  backstory: >
    You are an expert researcher at a top AI lab with 10 years
    of experience in time-series modeling and anomaly detection.
  model: claude-opus-4-5
  verbose: true
  max_iter: 5
  tools:
    - arxiv_search
    - web_scraper

writer:
  role: Technical Content Writer
  goal: Convert research findings into clear course material
  model: claude-sonnet-4-5
  verbose: false
  tools:
    - markdown_formatter
"""

config = yaml.safe_load(crewai_config)
print(f"Researcher model: {config['researcher']['model']}")
print(f"Researcher tools: {config['researcher']['tools']}")
print(f"Writer verbose: {config['writer']['verbose']}")

# The backstory folded scalar
backstory = config['researcher']['backstory']
print(f"\nBackstory (first 60 chars): {backstory[:60]}...")

In [ ]:
# YAML Anchors & Aliases — DRY principle
anchored_yaml = """
defaults: &model_defaults
  temperature: 0.7
  max_tokens: 2048
  top_p: 0.95

agents:
  researcher:
    <<: *model_defaults
    model: claude-opus-4-5
    temperature: 0.3   # Override just this field
  writer:
    <<: *model_defaults
    model: claude-sonnet-4-5
"""

cfg = yaml.safe_load(anchored_yaml)
print("Researcher:", cfg['agents']['researcher'])
print("Writer:", cfg['agents']['writer'])

### 🔴 Advanced: GitHub Actions YAML Structure

In [ ]:
# Build a GitHub Actions workflow programmatically
def build_ml_workflow(
    pipeline_script: str,
    python_version: str = "3.11",
    cron_schedule: str = "0 6 * * *",
    notify_slack: bool = True
) -> str:
    """Generate a GitHub Actions ML pipeline YAML."""
    
    workflow = {
        "name": "ML Daily Pipeline",
        "on": {
            "push": {"branches": ["main", "develop"]},
            "schedule": [{"cron": cron_schedule}]
        },
        "jobs": {
            "run-pipeline": {
                "runs-on": "ubuntu-latest",
                "steps": [
                    {"uses": "actions/checkout@v4"},
                    {
                        "name": "Setup Python",
                        "uses": "actions/setup-python@v5",
                        "with": {"python-version": python_version}
                    },
                    {"name": "Install deps", "run": "pip install -r requirements.txt"},
                    {
                        "name": "Run pipeline",
                        "run": f"python {pipeline_script}",
                        "env": {
                            "AWS_ACCESS_KEY_ID": "${{ secrets.AWS_ACCESS_KEY_ID }}",
                            "AWS_SECRET_ACCESS_KEY": "${{ secrets.AWS_SECRET_ACCESS_KEY }}"
                        }
                    },
                    {
                        "name": "Upload artifacts",
                        "uses": "actions/upload-artifact@v4",
                        "with": {"name": "pipeline-outputs", "path": "outputs/"}
                    }
                ]
            }
        }
    }
    
    return yaml.dump(workflow, default_flow_style=False, sort_keys=False)

print(build_ml_workflow("dt_hunter_daily.py"))

---
## 🔧 Part 3 — TOML
### 🟢 Beginner: TOML Basics

In [ ]:
# Python 3.11+ has tomllib built in!
import tomllib
import tomli_w  # pip install tomli-w for writing

toml_content = b"""
[pipeline]
name = "dt-hunter-daily"
version = "2.1.0"
schedule = "0 6 * * *"

[data]
s3_bucket = "marevlo-telemetry"
lookback_days = 7
min_records = 100

[model]
xgb_path = "models/hunter_xgb_v2.pkl"
anomaly_threshold = 0.75
features = ["cpu_util", "mem_util", "pfe_errors", "temp_c"]

[[agents]]
name = "researcher"
model = "claude-opus-4-5"

[[agents]]
name = "writer"
model = "claude-sonnet-4-5"
"""

config = tomllib.loads(toml_content.decode())

print(f"Pipeline: {config['pipeline']['name']} v{config['pipeline']['version']}")
print(f"S3 bucket: {config['data']['s3_bucket']}")
print(f"Threshold: {config['model']['anomaly_threshold']}")
print(f"Agents: {[a['name'] for a in config['agents']]}")

In [ ]:
# TOML native datetime support
toml_with_dates = b"""
[experiment]
name = "ablation-v3"
started_at = 2024-03-15T09:30:00Z
run_date = 2024-03-15
"""

exp = tomllib.loads(toml_with_dates.decode())
print(f"Started at: {exp['experiment']['started_at']}")
print(f"Type: {type(exp['experiment']['started_at'])}")  # Real datetime!
print(f"Run date type: {type(exp['experiment']['run_date'])}")  # Real date!

### 🟡 Intermediate: Pydantic-Validated TOML Config

In [ ]:
from pydantic import BaseModel, field_validator
from datetime import datetime

class XGBoostConfig(BaseModel):
    n_estimators: int
    max_depth: int
    learning_rate: float
    subsample: float = 0.8

    @field_validator('learning_rate')
    @classmethod
    def lr_must_be_valid(cls, v):
        if not (0 < v < 1):
            raise ValueError(f'learning_rate must be in (0,1), got {v}')
        return v

class DataConfig(BaseModel):
    s3_bucket: str
    lookback_days: int = 7
    min_records: int = 100
    features: list[str]

class PipelineConfig(BaseModel):
    name: str
    version: str
    data: DataConfig
    model: dict  # XGBoostConfig nested inside

# Load and validate
raw_toml = b"""
[data]
s3_bucket = "marevlo-telemetry"
lookback_days = 7
features = ["cpu_util", "mem_util", "pfe_errors"]

[model]
n_estimators = 500
max_depth = 6
learning_rate = 0.05
subsample = 0.8
"""

raw = tomllib.loads(raw_toml.decode())
xgb = XGBoostConfig.model_validate(raw['model'])
print(f"Validated XGBoost config: {xgb}")

# Test validation
try:
    bad = XGBoostConfig(n_estimators=100, max_depth=5, learning_rate=1.5)  # lr > 1!
except Exception as e:
    print(f"\nValidation caught error: {e}")

### 🔴 Advanced: Writing TOML for Experiment Tracking

In [ ]:
import tomli_w
from datetime import datetime, timezone

def save_experiment_toml(experiment: dict, path: str):
    """Save experiment config to TOML for reproducibility."""
    with open(path, 'wb') as f:
        tomli_w.dump(experiment, f)

experiment_config = {
    "experiment": {
        "name": "hunter-v2-dwt-ablation",
        "seed": 42,
        "tags": ["ablation", "production-candidate"],
    },
    "data": {
        "s3_path": "s3://marevlo-data/telemetry/2024-03/",
        "train_split": 0.8,
        "features": ["cpu_util", "mem_util", "pfe_errors", "temp_c"]
    },
    "model": {
        "xgboost": {
            "n_estimators": 500,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8
        },
        "isolation_forest": {
            "contamination": 0.05,
            "n_estimators": 200
        }
    },
    "thresholds": {
        "anomaly_score_min": 0.7,
        "alert_on_consecutive": 3
    }
}

import tempfile
with tempfile.NamedTemporaryFile(suffix='.toml', delete=False, mode='wb') as f:
    tomli_w.dump(experiment_config, f)
    path = f.name

# Verify by reading back
with open(path, 'rb') as f:
    verified = tomllib.load(f)

print(f"Saved & verified: {verified['experiment']['name']}")
print(f"XGBoost n_estimators: {verified['model']['xgboost']['n_estimators']}")

# Print the raw TOML
with open(path) as f:
    print("\n=== TOML File Content ===")
    print(f.read())

os.unlink(path)

---
## 📊 Part 4 — JSONL
### 🟢 Beginner: Reading & Writing JSONL

In [ ]:
import json, tempfile, os

# Create a JSONL file — LLM fine-tuning format
training_examples = [
    {
        "messages": [
            {"role": "system", "content": "You are a network reliability expert."},
            {"role": "user", "content": "What does a CPU utilization spike above 90% indicate?"},
            {"role": "assistant", "content": "CPU spikes above 90% on network linecards typically indicate route table churn, DDoS traffic, or software bugs causing excessive processing."}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a network reliability expert."},
            {"role": "user", "content": "What is PFE in Juniper routers?"},
            {"role": "assistant", "content": "PFE (Packet Forwarding Engine) is the ASIC-based hardware that performs high-speed packet forwarding in Juniper MX routers."}
        ]
    },
]

# Write JSONL
with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False) as f:
    for record in training_examples:
        f.write(json.dumps(record) + '\n')  # ← Newline separator is key!
    jsonl_path = f.name

# Read JSONL — one line at a time (memory efficient!)
print("Reading JSONL line by line:")
with open(jsonl_path) as f:
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        user_msg = next(m for m in record['messages'] if m['role'] == 'user')
        print(f"Example {i+1}: {user_msg['content'][:60]}...")

os.unlink(jsonl_path)

### 🟡 Intermediate: Agent Trace Logger

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

class AgentTraceLogger:
    """Logs every agent step as JSONL for debugging and auditing."""
    
    def __init__(self, session_id: str, log_dir: str = "/tmp/traces"):
        Path(log_dir).mkdir(parents=True, exist_ok=True)
        self.path = f"{log_dir}/{session_id}.jsonl"
        self.session_id = session_id

    def log(self, event_type: str, **kwargs):
        record = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "session_id": self.session_id,
            "type": event_type,
            **kwargs
        }
        with open(self.path, 'a') as f:
            f.write(json.dumps(record) + '\n')

    def replay(self) -> list[dict]:
        """Read all logged events."""
        with open(self.path) as f:
            return [json.loads(line) for line in f if line.strip()]

    def filter_events(self, event_type: str) -> list[dict]:
        """Get only specific event types."""
        return [e for e in self.replay() if e['type'] == event_type]

# Simulate an agent run
logger = AgentTraceLogger("hunter_sess_001")

logger.log("agent_start", task="Analyze network anomalies", devices=["JNP-001", "JNP-002"])
logger.log("tool_call", tool="get_telemetry", device="JNP-001", metric="cpu_util")
logger.log("tool_result", device="JNP-001", value=94.2, anomaly=True, score=0.91)
logger.log("reasoning", thought="JNP-001 CPU at 94.2% with score 0.91. Checking memory next.")
logger.log("tool_call", tool="get_telemetry", device="JNP-001", metric="mem_util")
logger.log("tool_result", device="JNP-001", value=67.3, anomaly=False, score=0.32)
logger.log("final_answer", risk="HIGH", recommendation="Immediate maintenance on JNP-001", confidence=0.89)

# Replay and analyze
all_events = logger.replay()
tool_calls = logger.filter_events("tool_call")
anomalies = [e for e in logger.filter_events("tool_result") if e.get('anomaly')]

print(f"Total events logged: {len(all_events)}")
print(f"Tool calls made: {len(tool_calls)}")
print(f"Anomalies detected: {len(anomalies)}")
print(f"\nFinal recommendation:")
final = logger.filter_events("final_answer")[0]
print(f"  Risk: {final['risk']} | Confidence: {final['confidence']}")
print(f"  Action: {final['recommendation']}")

### 🔴 Advanced: Memory-Efficient Large File Processing

In [ ]:
import json
from typing import Iterator

def stream_jsonl(path: str, batch_size: int = 100) -> Iterator[list[dict]]:
    """Stream JSONL file in batches — constant memory regardless of file size."""
    batch = []
    with open(path) as f:
        for line in f:
            if line.strip():
                batch.append(json.loads(line))
                if len(batch) >= batch_size:
                    yield batch
                    batch = []
    if batch:
        yield batch  # Final partial batch

# Generate a large synthetic dataset
import random, tempfile

large_jsonl_path = "/tmp/large_telemetry.jsonl"
N = 10_000

print(f"Generating {N} telemetry records...")
with open(large_jsonl_path, 'w') as f:
    for i in range(N):
        record = {
            "device_id": f"JNP-{i % 50:03d}",
            "cpu_util": round(random.uniform(20, 99), 2),
            "mem_util": round(random.uniform(30, 95), 2),
            "pfe_errors": random.randint(0, 100),
            "timestamp": 1710480000 + i * 60
        }
        f.write(json.dumps(record) + '\n')

# Process in batches — peak memory ≈ batch_size records, not 10k!
total_processed = 0
high_cpu_alerts = 0
batches = 0

for batch in stream_jsonl(large_jsonl_path, batch_size=500):
    batches += 1
    total_processed += len(batch)
    high_cpu_alerts += sum(1 for r in batch if r['cpu_util'] > 90)

print(f"Processed {total_processed} records in {batches} batches")
print(f"High CPU alerts (>90%): {high_cpu_alerts}")
os.unlink(large_jsonl_path)

---
## 🚀 Part 5 — Full Integration: Hunter Pipeline Pattern

All 4 formats working together in one pipeline.

In [ ]:
import json, yaml, tomllib, tomli_w
from datetime import datetime, timezone
from pathlib import Path

# === STEP 1: TOML — Load pipeline config ===
toml_config_bytes = b"""
[pipeline]
name = "dt-hunter-demo"
version = "2.1.0"

[model]
anomaly_threshold = 0.75
features = ["cpu_util", "mem_util", "pfe_errors"]

[alerts]
min_severity = "HIGH"
"""

config = tomllib.loads(toml_config_bytes.decode())
print(f"[TOML] Loaded: {config['pipeline']['name']} v{config['pipeline']['version']}")

# === STEP 2: YAML — Agent config ===
agent_yaml = """
analyst:
  role: Network Reliability Agent
  model: claude-opus-4-5
  temperature: 0.2
  tools: [get_telemetry, run_anomaly_detection, send_alert]
"""
agent_config = yaml.safe_load(agent_yaml)
print(f"[YAML] Agent: {agent_config['analyst']['role']}")

# === STEP 3: JSON — Simulate LLM tool schema ===
tool_schema = {
    "name": "run_anomaly_detection",
    "input_schema": {
        "type": "object",
        "properties": {
            "device_ids": {"type": "array", "items": {"type": "string"}},
            "threshold": {"type": "number"}
        },
        "required": ["device_ids"]
    }
}
print(f"[JSON] Tool schema: {tool_schema['name']} ({len(tool_schema['input_schema']['properties'])} params)")

# === STEP 4: JSONL — Log execution ===
log_path = "/tmp/hunter_demo_run.jsonl"

def log_event(event_type: str, **kwargs):
    record = {"ts": datetime.now(timezone.utc).isoformat(), "type": event_type, **kwargs}
    with open(log_path, 'a') as f:
        f.write(json.dumps(record) + '\n')

# Simulate pipeline run
log_event("pipeline_start", config=config['pipeline']['name'], devices=5)

# Mock results
results = [
    {"device": "JNP-001", "score": 0.91, "anomaly": True},
    {"device": "JNP-002", "score": 0.23, "anomaly": False},
    {"device": "JNP-003", "score": 0.82, "anomaly": True},
]

for r in results:
    log_event("device_scored", **r)

anomalies = [r for r in results if r['anomaly']]
log_event("pipeline_end", total=len(results), anomalies=len(anomalies))

# Read back log
with open(log_path) as f:
    events = [json.loads(line) for line in f]

print(f"[JSONL] Logged {len(events)} events, found {len(anomalies)} anomalies")

print("\n=== Final Summary ===")
print(f"Format: TOML → config | YAML → agents | JSON → tools | JSONL → logs")
print(f"Devices analyzed: {len(results)} | High-risk devices: {len(anomalies)}")
os.unlink(log_path)

---
## 🏆 Module Challenge

Build a mini pipeline that:
1. Reads an agent config from YAML
2. Reads model hyperparameters from TOML
3. Constructs a JSON tool schema for `predict_failure(device_id, threshold)`
4. Runs a mock prediction on 5 fake devices and logs results as JSONL
5. Reads the JSONL log and prints a summary report

**Bonus:** Add Pydantic validation for the TOML config.

In [ ]:
# Your solution here!
# Hints:
# - Use yaml.safe_load() for YAML
# - Use tomllib.loads() for TOML (bytes or .decode())
# - Build tool schema as a Python dict, then json.dumps()
# - Write logs with open(path, 'a') + json.dumps() + '\n'

# === STEP 1: Define your YAML agent config ===
agent_yaml = """
# Write your agent YAML here
"""

# === STEP 2: Define your TOML config ===
toml_config = b"""
# Write your TOML here
"""

# === STEP 3: Build JSON tool schema ===
# ...

# === STEP 4: Run mock predictions and log ===
# ...

# === STEP 5: Read log and summarize ===
# ...

---
## 📚 Module Summary

| Format | Best For | Key Python Library |
|--------|----------|--------------------|
| **JSON** | LLM APIs, tool schemas, inter-service comms | `json`, `orjson` |
| **YAML** | CI/CD pipelines, agent configs, K8s | `pyyaml` (safe_load!) |
| **TOML** | Project config, ML experiments, packages | `tomllib` (stdlib 3.11+) |
| **JSONL** | Training data, execution logs, eval sets | `json` line-by-line |

### Critical Rules
- Always use `yaml.safe_load()`, never `yaml.load()`
- YAML `NO`/`YES` → booleans. Quote strings that could be misread.
- TOML dates are native datetime objects — use this!
- JSONL: one valid JSON object per line, no wrapping array
- Use `orjson` when JSON serialization is a bottleneck

---
**Next Module → JSON Schema & Structured LLM Outputs**